<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

# SimCLR על STL-10 — מחקר אוגמנטציות ויעילות תוויות

## מטרת המחקר

המחקר בוחן כיצד ארבע מדיניות אוגמנטציה משפיעות על איכות הייצוגים הנלמדים
באימון SimCLR. כל המדיניות משתמשות באותה ארכיטקטורה, באותו seed, באותו
תקציב של 100 epochs ובאותו פרוטוקול אופטימיזציה.

המחקר כולל שלושה שלבים נפרדים:

1. **אימון ייצוגים ללא תוויות** — כל מדיניות מאומנת על 100,000 תמונות
   ה־`unlabeled` של STL-10.
2. **בחירת מדיניות, checkpoint ו־\(C\)** — ההחלטות מתקבלות רק מתוך
   ה־`labeled train`, באמצעות עשרת ניסויי האימון הרשמיים של STL-10.
3. **הערכת יעילות תוויות** — לאחר נעילת כל ההחלטות, מאמנים עשרה מסווגים
   לינאריים נפרדים. כל מסווג משתמש ב־1,000 התמונות של fold רשמי אחד
   ונבדק על כל 8,000 תמונות ה־`test`.


ה־benchmark הסופי משתמש בעשרת ה־folds הרשמיים כפי שהם מגיעים:
1,000 תמונות לאימון מסווג לינארי מול 8,000 תמונות `test`.

</div>

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 1. ספריות ובדיקת סביבת ההרצה

התא מייבא את הספריות הדרושות לעיבוד הנתונים, בניית המודל, אימון SimCLR,
הערכת הייצוגים, שמירת התוצאות והצגתן במחברת.

ההנחה היא שהמחברת מורצת בסביבה GPU(אחרת היא לא תוכל לרוץ)
התא מגדיר backend דטרמיניסטי ל-pytortch כחלק מהרצון להשוואה הוגנת.

</div>

In [ ]:
from pathlib import Path
import copy
import gc
import json
import math
import random
import shutil

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss
)
from sklearn.model_selection import (StratifiedKFold, train_test_split,)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 2. הגדרות ניסוי ה-simClr

התא מרכז את כל הקבועים שנשתמש בהם במהלך הניסוים של simClr

בנוסף, מוגדרת פונקצייה הדואגת ל-seed  קבועים בכדי שכלל ההשוואות יהיו הוגנות

</div>

In [ ]:
SEED = 42

IMAGE_SIZE = 96
NUMBER_OF_CLASSES = 10

BATCH_SIZE = 256
FEATURE_BATCH_SIZE = 512
NUM_WORKERS = 4

EPOCHS = 100
WARMUP_EPOCHS = 10
PROBE_EPOCHS = 50, 60, 70, 80, 90, 100
TEMPERATURE = 0.20
MAX_LEARNING_RATE = 5e-4
MIN_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4


STEM_WIDTH = 48
STAGE_DEPTHS = 2, 3, 4, 3
STAGE_WIDTHS = 64, 128, 256, 384
ENCODER_DIM = 512
PROJECTOR_HIDDEN_DIM = 512
PROJECTOR_OUTPUT_DIM = 128
SE_REDUCTION = 16

LINEAR_C_GRID = 0.1, 1.0, 10.0, 100.0
CV_SPLITS = 5
LINEAR_MAX_ITER = 5000

RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
PROJECT_ROOT = Path("/content/SimCLR_STL10_Label_Efficiency")
DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results" / RUN_ID
CHECKPOINTS_ROOT = RESULTS_ROOT / "checkpoints"
HISTORIES_ROOT = RESULTS_ROOT / "histories"
PROBES_ROOT = RESULTS_ROOT / "development_probes"
FINAL_ROOT = RESULTS_ROOT / "final_benchmark"
CLASSIFIERS_ROOT = FINAL_ROOT / "linear_classifiers"

for folder in (DATA_ROOT,
    CHECKPOINTS_ROOT,
    HISTORIES_ROOT,
    PROBES_ROOT,
    FINAL_ROOT,
    CLASSIFIERS_ROOT
    ):
    folder.mkdir(parents=True, exist_ok=True)


def reset_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


def seeded_generator():
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return generator


def save_csv(frame, path):
    frame.to_csv(path, index=False, float_format="%.8f")


reset_seed()

print("Run ID:", RUN_ID)
print("Results:", RESULTS_ROOT)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 3. טעינת הנתונים וחישוב סטטיסטיקות הנרמול

נטענים שני סטים:

- `unlabeled` — 100,000 התמונות המשמשות לאימון SimCLR;
- `train` — 5,000 התמונות המתויגות, המשמשות בהמשך ל־development
  ול־benchmark הלינארי.

הממוצע וסטיית התקן מחושבים **רק** מתוך ה־`unlabeled split`, לפני
אוגמנטציות ובנפרד לכל ערוץ RGB.

החישוב מתבצע ב־chunks:

1. כל chunk מומר ל־`float64`.
2. נצברים סכום הפיקסלים וסכום ריבועי הפיקסלים לכל ערוץ.
3. הממוצע מחושב מן הסכום הכולל.
4. השונות מחושבת באמצעות \(E[X^2]-E[X]^2\).
5. הערכים מותאמים לטווח \([0,1]\) באמצעות חלוקה ב־255.

הסיבה לחישוב בתצורה הזאת , chunks וחישוב מומנטים הוא להמנע מאפשרות של הצפת הזיכרון.

</div>

In [ ]:
raw_unlabeled = datasets.STL10(root=DATA_ROOT, split="unlabeled", download=True)
raw_labeled_train = datasets.STL10(root=DATA_ROOT, split="train", download=True)


def compute_channel_statistics(data, chunk_size=512):
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_square_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0

    for start in tqdm(range(0, len(data), chunk_size), desc="Compute normalization"):
        end = min(start + chunk_size, len(data))
        chunk = data[start:end].astype(np.float64, copy=True)

        channel_sum += chunk.sum(axis=(0, 2, 3))
        channel_square_sum += np.einsum("nchw,nchw->c",
                                        chunk,
                                        chunk,
                                        optimize=True)
        pixel_count += (chunk.shape[0] * chunk.shape[2] * chunk.shape[3])

    mean = channel_sum / pixel_count / 255.0
    second_moment = (channel_square_sum / pixel_count / (255.0 ** 2))
    variance = second_moment - np.square(mean)
    standard_deviation = np.sqrt(np.maximum(variance, 0.0))

    return mean.tolist(), standard_deviation.tolist()


SSL_MEAN, SSL_STD = compute_channel_statistics(raw_unlabeled.data)

print("Mean:", SSL_MEAN)
print("Standard deviation:", SSL_STD)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 4. עשרת ניסויי האימון הרשמיים וחלוקות ה־development

הקוד קורא ישירות את `fold_indices.txt` של STL-10. הקובץ מגדיר עשרה
ניסויי אימון רשמיים, ובכל אחד מהם 1,000 אינדקסים מתוך 5,000 תמונות
ה־`train`.

### שלב ה־development
בשלב האימון של ה-encoder אנחנו נרצה לשמור את ה-best model שאחר כך נוכל להקפיא את ה-encoder ולהשתמש בו עבור feature ectracor ל-logistic regression.
בשביל זה אנחנו נבצע אחת לכמה זמן linear probbing על ידי הקפאה זמנית של ה-encoder
בחירת c טוב ל- probbing שישמש אותנו אחר כך גם לשלב האחרון
ובעצם ניסיון לראות את יכולת הסיווג על ידי הוקטורים במרחב הנסתר שהם תוצאה של ה-encoder.
לשם כך נבצע חלוקה של 5 flods בעלי 800 דוגמאות אימון ו=200 ולידציה.
פרטים נוספים בסיכום המחקר.


### ה־benchmark הסופי

לאחר סיום הבחירה לא משתמשים עוד בחלוקת 800/200. כל אחד מעשרת המסווגים
הסופיים מאומן על כל 1,000 התמונות של ה־fold הרשמי המתאים.

</div>

In [ ]:
fold_file = DATA_ROOT / "stl10_binary" / "fold_indices.txt"

STL10_OFFICIAL_FOLDS = np.stack([
    np.fromstring(line, dtype=np.int64, sep=" ")
    for line in fold_file.read_text(encoding="utf-8").strip().splitlines()
    ])

FULL_TRAIN_LABELS = np.asarray(raw_labeled_train.labels, dtype=np.int64)

DEVELOPMENT_SPLITS = {}

for fold_index, official_indices in enumerate(STL10_OFFICIAL_FOLDS):
    fold_positions = np.arange(len(official_indices))
    fold_labels = FULL_TRAIN_LABELS[official_indices]

    train_positions, validation_positions = train_test_split(
        fold_positions,
        test_size=0.20,
        random_state= SEED + fold_index,
        stratify=fold_labels
        )


    DEVELOPMENT_SPLITS[fold_index] = train_positions, validation_positions

print("Official training folds:", STL10_OFFICIAL_FOLDS.shape)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 5. מדיניות האוגמנטציה ו־DataLoaders

`TwoViews` יוצר שתי תצוגות שונות על ידי דגימה אקראית של אוגמנטציות מאותה מדיניות


| מדיניות | סדר הפעולות |
|---|---|
| `P1_crop_color` | `RandomResizedCrop → ColorJitter → ToTensor → Normalize` |
| `P2_crop_color_blur` | `RandomResizedCrop → ColorJitter → GaussianBlur → ToTensor → Normalize` |
| `P3_crop_flip_color` | `RandomResizedCrop → RandomHorizontalFlip → ColorJitter → ToTensor → Normalize` |
| `P4_full_policy` | `RandomResizedCrop → RandomHorizontalFlip → ColorJitter → RandomGrayscale → GaussianBlur → ToTensor → Normalize` |

### פרמטרים קבועים

- `RandomResizedCrop`: גודל פלט 96×96, `scale=(0.20, 1.00)`,
  `ratio=(0.75, 4/3)`.
- `ColorJitter`: עוצמה 0.40 ל־brightness, contrast ו־saturation,
   0.10 ל־hue, והפעלה בהסתברות 0.80.
- `RandomHorizontalFlip`: הסתברות 0.50.
- `RandomGrayscale`: הסתברות 0.20.
- `GaussianBlur`: kernel בגודל 9, `sigma=(0.10, 2.00)`,
  והפעלה בהסתברות 0.50.

`EVALUATION_TRANSFORM` אינו כולל אוגמנטציה אקראית; הוא מבצע רק
`ToTensor` ו־`Normalize`.
</div>

In [ ]:
class TwoViews:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, image):
        return (self.transform(image), self.transform(image))


CROP_ARGUMENTS = {
    "size": IMAGE_SIZE,
    "scale": (0.20, 1.00),
    "ratio": (0.75, 4 / 3),
    "antialias": True
}

COLOR_JITTER_ARGUMENTS = {
    "brightness": 0.40,
    "contrast": 0.40,
    "saturation": 0.40,
    "hue": 0.10
    }


def color_jitter_operation():
    return transforms.RandomApply([transforms.ColorJitter(**COLOR_JITTER_ARGUMENTS)], p=0.80)


def gaussian_blur_operation():
    return transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=9, sigma=(0.10, 2.00))], p=0.50)


def normalization():
    return [transforms.ToTensor(), transforms.Normalize(mean=SSL_MEAN, std=SSL_STD)]


POLICY_TRANSFORMS = {
    "P1_crop_color": transforms.Compose([
        transforms.RandomResizedCrop(**CROP_ARGUMENTS),
        color_jitter_operation(),
        *normalization()
    ]),
    "P2_crop_color_blur": transforms.Compose([
        transforms.RandomResizedCrop(**CROP_ARGUMENTS),
        color_jitter_operation(),
        gaussian_blur_operation(),
        *normalization()
    ]),
    "P3_crop_flip_color": transforms.Compose([
        transforms.RandomResizedCrop(**CROP_ARGUMENTS),
        transforms.RandomHorizontalFlip(p=0.50),
        color_jitter_operation(),
        *normalization()
    ]),
    "P4_full_policy": transforms.Compose([
        transforms.RandomResizedCrop(**CROP_ARGUMENTS),
        transforms.RandomHorizontalFlip(p=0.50),
        color_jitter_operation(),
        transforms.RandomGrayscale(p=0.20),
        gaussian_blur_operation(),
        *normalization()
    ])
}

EVALUATION_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=SSL_MEAN,std=SSL_STD)
    ])


def make_ssl_loader(policy_name):
    dataset = copy.copy(raw_unlabeled)
    dataset.transform = TwoViews(POLICY_TRANSFORMS[policy_name])

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        generator=seeded_generator()
        )


labeled_train_dataset = datasets.STL10(root=DATA_ROOT, split="train",transform=EVALUATION_TRANSFORM)


POLICY_PRIORITY = {policy_name: priority for priority, policy_name in enumerate(POLICY_TRANSFORMS)}

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 6. ארכיטקטורת SimCLR

מודל שלם המכיל את ה-encoder וה-projection head.
בקובץ המחקר ישנו הסבר מעמיק על הארכיטקטורה.

</div>

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction):
        super().__init__()

        hidden_channels = max(channels // reduction, 16)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.scale = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, channels, kernel_size=1),
            nn.Sigmoid()
            )

    def forward(self, x):
        channel_weights = self.scale(self.pool(x))
        return x * channel_weights


class PreActivationSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()

        self.norm1 = nn.BatchNorm2d(in_channels)
        self.activation1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            padding_mode="zeros",
            bias=False,
        )

        self.norm2 = nn.BatchNorm2d(out_channels)
        self.activation2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            padding_mode="zeros",
            bias=False,
        )

        self.se = SqueezeExcitation(out_channels, SE_REDUCTION)

        needs_projection = stride != 1 or in_channels != out_channels
        self.downsample = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
            if needs_projection
            else None
            )

    def forward(self, x):
        preactivated = self.activation1(self.norm1(x))
        shortcut = x if self.downsample is None else self.downsample(preactivated)

        residual = self.conv1(preactivated)
        residual = self.norm2(residual)
        residual = self.activation2(residual)
        residual = self.conv2(residual)
        residual = self.se(residual)

        return shortcut + residual


class SEResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                3,
                STEM_WIDTH,
                kernel_size=3,
                stride=2,
                padding=1,
                padding_mode="zeros",
                bias=False,
            ),
            nn.BatchNorm2d(STEM_WIDTH),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                STEM_WIDTH,
                STAGE_WIDTHS[0],
                kernel_size=3,
                stride=1,
                padding=1,
                padding_mode="zeros",
                bias=False,
            ),
        )

        stages = []
        in_channels = STAGE_WIDTHS[0]

        for stage_index, (depth, out_channels) in enumerate(zip(STAGE_DEPTHS, STAGE_WIDTHS)):
            blocks = []

            for block_index in range(depth):
                stride = 2 if stage_index > 0 and block_index == 0 else 1
                blocks.append(PreActivationSEBlock(in_channels, out_channels, stride))
                in_channels = out_channels

            stages.append(nn.Sequential(*blocks))

        self.stages = nn.Sequential(*stages)
        self.final_norm = nn.BatchNorm2d(in_channels)
        self.final_activation = nn.ReLU(inplace=True)
        self.output_projection = nn.Conv2d(in_channels, ENCODER_DIM, kernel_size=1, bias=False)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        x = self.final_norm(x)
        x = self.final_activation(x)
        x = self.output_projection(x)
        x = self.global_pool(x)
        return x.flatten(1)


class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(ENCODER_DIM, PROJECTOR_HIDDEN_DIM, bias=False),
            nn.BatchNorm1d(PROJECTOR_HIDDEN_DIM),
            nn.ReLU(inplace=True),
            nn.Linear(PROJECTOR_HIDDEN_DIM, PROJECTOR_OUTPUT_DIM, bias=False)
            )

    def forward(self, x):
        return self.layers(x)


def initialize_model_weights(model):
    se_reduce_ids = set()
    se_expand_ids = set()

    for module in model.modules():
        if isinstance(module, SqueezeExcitation):
            se_reduce_ids.add(id(module.scale[0]))
            se_expand_ids.add(id(module.scale[2]))

    output_projection_id = id(model.encoder.output_projection)
    projector_hidden_id = id(model.projection_head.layers[0])
    projector_output_id = id(model.projection_head.layers[3])

    for module in model.modules():
        module_id = id(module)

        if isinstance(module, nn.Conv2d):
            if module_id in se_reduce_ids:
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
            elif module_id in se_expand_ids or module_id == output_projection_id:
                nn.init.xavier_uniform_(module.weight, gain=1.0)
            else:
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Linear):
            if module_id == projector_hidden_id:
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
            elif module_id == projector_output_id:
                nn.init.xavier_uniform_(module.weight, gain=1.0)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)


class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = SEResNetEncoder()
        self.projection_head = ProjectionHead()
        initialize_model_weights(self)

    def forward(self, x):
        representation = self.encoder(x)
        return self.projection_head(representation)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif">

## 7. פונקציית ההפסד NT-Xent

הפונקציה מקבלת שני batches של projections

תחילה היא מסדרת את שתי ה־views של כל תמונה כזוגות רצופים

לאחר מכן:

1. כל projection עובר נרמול \(L_2\), ולכן המכפלה הפנימית מייצגת
   cosine similarity.
2. מטריצת הדמיון מחולקת ב־temperature.
3. `indices ^ 1` ממפה כל anchor ל־positive המתאים לו.
4. דמיון של view לעצמה מוסר מן המכנה באמצעות \(-\infty\).
5. ה־positive נשאר במכנה; יתר ה־views שאינן self ואינן positive
   משמשות negatives.

לכל anchor מחושב:

$$
-\frac{\operatorname{sim}(z_i,z_{j(i)})}{\tau}
+
\log\sum_{k\neq i}\exp\left(\frac{\operatorname{sim}(z_i,z_k)}{\tau}\right)
$$

ה־loss הסופי הוא הממוצע על כל \(2B\) ה־anchors.

</div>

In [ ]:
def nt_xent(z1, z2):
    batch_size = z1.shape[0]

    projections = torch.stack((z1, z2), dim=1,).reshape(2 * batch_size, -1)
    projections = F.normalize(projections, p=2, dim=1)

    logits = (projections @ projections.T) / TEMPERATURE

    number_of_views = 2 * batch_size
    indices = torch.arange(number_of_views, device=projections.device)
    positive_indices = indices ^ 1

    self_mask = torch.eye(number_of_views, dtype=torch.bool, device=projections.device)

    logits_without_self = logits.masked_fill(self_mask, -torch.inf)

    positive_logits = logits[indices, positive_indices]

    per_anchor_loss = (-positive_logits + torch.logsumexp(logits_without_self, dim=1))

    return per_anchor_loss.mean()

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 8. Linear Probe בשלב ה־development

מטרת התא היא להעריך כל checkpoint מבלי להשתמש ב־test.

### הפקת features

האנקודר מופעל ב־`eval()` ותחת `torch.inference_mode()`. לכל 5,000
תמונות ה־`labeled train` מופק פעם אחת וקטור \(h\) בגודל 512 באמצעות
transform דטרמיניסטי.

### בחירת \(C\) עבור checkpoint

לכל אחד מעשרת ה־folds:

1. נשלפות 800 תמונות ה־development-train.
2. עבור כל \(C\in\{0.1,1,10,100\}\) מתבצע `StratifiedKFold` עם
   חמש חלוקות בתוך אותן 800 תמונות בלבד.
3. בכל split מותאם מחדש ה־Pipeline המלא:
   `StandardScaler → LogisticRegression`.
4. לכל \(C\) נשמרים mean Accuracy ו־mean Macro-F1 של ה־CV הפנימי.

לאחר עשרת ה־folds, התוצאות מאוגדות לפי \(C\). לכל ערך מחושבים
הממוצעים על פני כל ה־folds. נבחר **\(C\) יחיד עבור ה־checkpoint כולו**
לפי הסדר:

1. Accuracy ממוצע גבוה יותר;
2. Macro-F1 ממוצע גבוה יותר;
3. במקרה של שוויון מלא — \(C\) קטן יותר.

### הערכת ה־checkpoint

אותו \(C\) נבחר משמש בכל עשרת ה־folds:

- ה־Pipeline מותאם מחדש על 800 תמונות ה־development-train;
- ההערכה מתבצעת על 200 תמונות ה־development-validation.

סיכום ה־checkpoint כולל:

- `selected_c`;
- mean development Accuracy;
- mean development Macro-F1;
- mean development Log Loss.

`is_better_checkpoint()` משווה checkpoints לפי Accuracy ולאחר מכן
Macro-F1. בשוויון מלא ה־checkpoint הקודם נשאר, ולכן בתוך אותה policy
נשמר ה־epoch המוקדם יותר.

</div>

In [ ]:
@torch.inference_mode()
def extract_features(encoder, dataset):
    loader = DataLoader(
        dataset,
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
        )

    feature_batches = []
    label_batches = []
    encoder.eval()

    for images, labels in tqdm(loader, leave=False, desc="Extract features"):
        features = encoder(images.to(DEVICE, non_blocking=True))
        feature_batches.append(features.cpu().numpy())
        label_batches.append(labels.numpy())

    return np.concatenate(feature_batches), np.concatenate(label_batches)


def make_linear_classifier(c_value):
    return Pipeline([
        ("scaler", StandardScaler()),
         ("classifier",
          LogisticRegression(C=c_value,solver="lbfgs",max_iter=LINEAR_MAX_ITER))
         ])


def classification_metrics(labels, predictions, probabilities):
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
        "log_loss": log_loss(labels, probabilities, labels=np.arange(NUMBER_OF_CLASSES))
         }


def cross_validate_c_values(features, labels, random_state):
    splitter = StratifiedKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=random_state
    )

    rows = []

    for c_value in LINEAR_C_GRID:
        fold_log_losses = []
        fold_accuracies = []
        fold_f1_scores = []

        for train_indices, validation_indices in splitter.split(features, labels):
            classifier = make_linear_classifier(c_value)
            classifier.fit(features[train_indices], labels[train_indices])

            validation_features = features[validation_indices]
            validation_labels = labels[validation_indices]

            predictions = classifier.predict(validation_features)
            probabilities = classifier.predict_proba(validation_features)

            fold_log_losses.append(
                log_loss(
                    validation_labels,
                    probabilities,
                    labels=np.arange(NUMBER_OF_CLASSES)
                )
            )

            fold_accuracies.append(
                accuracy_score(validation_labels, predictions)
            )

            fold_f1_scores.append(
                f1_score(validation_labels, predictions, average="macro", zero_division=0)
            )

        rows.append({
            "c": float(c_value),
            "log_loss_mean": float(np.mean(fold_log_losses)),
            "accuracy_mean": float(np.mean(fold_accuracies)),
            "f1_macro_mean": float(np.mean(fold_f1_scores))
        })

    return pd.DataFrame(rows)


def evaluate_development_folds(encoder):
    features, labels = extract_features(encoder, labeled_train_dataset)

    prepared_folds = []
    c_search_frames = []

    for fold_index, official_fold_indices in enumerate(STL10_OFFICIAL_FOLDS):
        fold_number = fold_index + 1
        train_positions, validation_positions = DEVELOPMENT_SPLITS[fold_index]

        fold_features = features[official_fold_indices]
        fold_labels = labels[official_fold_indices]

        development_train_features = fold_features[train_positions]
        development_train_labels = fold_labels[train_positions]
        development_validation_features = fold_features[validation_positions]
        development_validation_labels = fold_labels[validation_positions]

        c_search = cross_validate_c_values(
            development_train_features,
            development_train_labels,
            random_state=SEED + fold_index
        )

        c_search.insert(0, "fold", fold_number)
        c_search_frames.append(c_search)

        prepared_folds.append({
            "fold": fold_number,
            "train_features": development_train_features,
            "train_labels": development_train_labels,
            "validation_features": development_validation_features,
            "validation_labels": development_validation_labels
        })

    c_search_results = pd.concat(c_search_frames, ignore_index=True)

    c_summary = (
        c_search_results
        .groupby("c", as_index=False)
        .agg(
            log_loss_mean_across_folds=("log_loss_mean", "mean"),
            accuracy_mean_across_folds=("accuracy_mean", "mean"),
            f1_macro_mean_across_folds=("f1_macro_mean", "mean")
        )
        .sort_values(
            by=[
                "log_loss_mean_across_folds",
                "accuracy_mean_across_folds",
                "f1_macro_mean_across_folds",
                "c"
            ],
            ascending=[True, False, False, True]
        )
        .reset_index(drop=True)
    )

    selected_c = float(c_summary.iloc[0]["c"])
    c_summary["selected"] = c_summary["c"].eq(selected_c)

    c_search_results = c_search_results.merge(
        c_summary,
        on="c",
        how="left"
    )

    fold_rows = []

    for fold in prepared_folds:
        classifier = make_linear_classifier(selected_c)
        classifier.fit(fold["train_features"], fold["train_labels"])

        predictions = classifier.predict(fold["validation_features"])
        probabilities = classifier.predict_proba(fold["validation_features"])

        metrics = classification_metrics(
            fold["validation_labels"],
            predictions,
            probabilities
        )

        fold_rows.append({
            "fold": fold["fold"],
            "c": selected_c,
            **metrics
        })

    fold_results = pd.DataFrame(fold_rows)

    summary = {
        "selected_c": selected_c,
        "log_loss_mean": float(fold_results["log_loss"].mean()),
        "accuracy_mean": float(fold_results["accuracy"].mean()),
        "f1_macro_mean": float(fold_results["f1_macro"].mean())
    }

    return summary, fold_results, c_search_results


def is_better_result(candidate, current_best, loss_key, accuracy_key, f1_key,
                     loss_epsilon=1e-4, metric_epsilon=5e-4):
    if current_best is None:
        return True

    loss_improvement = current_best[loss_key] - candidate[loss_key]
    if loss_improvement > loss_epsilon:
        return True
    if loss_improvement < -loss_epsilon:
        return False

    accuracy_improvement = candidate[accuracy_key] - current_best[accuracy_key]
    if accuracy_improvement > metric_epsilon:
        return True
    if accuracy_improvement < -metric_epsilon:
        return False

    f1_improvement = candidate[f1_key] - current_best[f1_key]
    return f1_improvement > metric_epsilon

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 9. אופטימיזציה ואימון כל מדיניות

### AdamW ו־weight decay

הפרמטרים מחולקים לשתי קבוצות:

- פרמטרים דו־ממדיים ומעלה שאינם biases מקבלים
  `weight_decay=1e-4`;
- כל פרמטר חד־ממדי וכל פרמטר ששמו מסתיים ב־`.bias` מקבל
  `weight_decay=0`.

לכן משקלי Conv ו־Linear עוברים weight decay, בעוד שפרמטרי BatchNorm
וכל ה־biases, לרבות biases של מודולי SE, מוחרגים ממנו.

### לוח קצב הלמידה

ה־optimizer מאותחל ב־learning rate אפס, אך לפני כל `optimizer.step`
נקבע learning rate מפורש:

- במשך 10 epochs מתבצע Linear Warmup עד \(5\times10^{-4}\);
- לאחר מכן מתבצע Cosine Decay עד \(10^{-5}\);
- העדכון מתבצע בכל optimizer step, לא רק בסוף epoch.

### אימון epoch

בכל batch:

1. שתי ה־views מועברות ל־GPU.
2. ה־views מחוברות ל־batch אחד ועוברות דרך האנקודר וה־Projection Head.
3. הפלט מפוצל בחזרה ל־\(z_1\) ול־\(z_2\).
4. מחושב NT-Xent.
5. מתבצעים backpropagation ועדכון AdamW.

היסטוריית כל epoch כוללת את ה־learning rate בתחילתו ובסופו ואת
ממוצע NT-Xent המשוקלל לפי מספר התמונות המקוריות.

### Probes ושמירת checkpoints

Probe מתבצע רק ב־epochs ‏50, 60, 70, 80, 90 ו־100. לכל probe נשמרים:

- תוצאות עשרת ה־development folds;
- תוצאות חיפוש \(C\);
- סיכום ה־checkpoint.

לכל policy נשמרים:

- `best_encoder.pt` — ה־checkpoint הטוב ביותר לפי development Accuracy
  ולאחר מכן Macro-F1;
- `epoch_100_encoder.pt` — האנקודר בסיום התקציב הקבוע;
- היסטוריית האימון;
- היסטוריית כל ה־probes.

האימון ממשיך תמיד עד epoch 100; אין Early Stopping.

</div>

In [ ]:
def build_optimizer(model):
    decay_parameters = []
    no_decay_parameters = []

    for parameter in model.parameters():
        if parameter.ndim == 1:
            no_decay_parameters.append(parameter)
        else:
            decay_parameters.append(parameter)

    return torch.optim.AdamW(
        [
            {"params": decay_parameters, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay_parameters, "weight_decay": 0.0}
        ],
        lr=0.0,
    )


def learning_rate_at_step(step, total_steps, warmup_steps):
    if step < warmup_steps:
        warmup_progress = (step + 1) / warmup_steps
        return MAX_LEARNING_RATE * warmup_progress

    cosine_denominator = max(total_steps - warmup_steps - 1, 1)
    cosine_progress = min((step - warmup_steps) / cosine_denominator, 1.0)
    cosine_factor = 0.5 * (1.0 + math.cos(math.pi * cosine_progress))
    return MIN_LEARNING_RATE + (MAX_LEARNING_RATE - MIN_LEARNING_RATE) * cosine_factor


def set_learning_rate(optimizer, learning_rate):
    for parameter_group in optimizer.param_groups:
        parameter_group["lr"] = learning_rate


def train_epoch(model, loader, optimizer, global_step, total_steps, warmup_steps):
    model.train()

    total_loss = 0.0
    sample_count = 0
    first_learning_rate = None
    last_learning_rate = None

    for (view1, view2), _ in tqdm(loader, leave=False, desc="Train"):
        current_learning_rate = learning_rate_at_step(global_step, total_steps, warmup_steps)
        set_learning_rate(optimizer, current_learning_rate)

        if first_learning_rate is None:
            first_learning_rate = current_learning_rate
        last_learning_rate = current_learning_rate

        view1 = view1.to(DEVICE, non_blocking=True)
        view2 = view2.to(DEVICE, non_blocking=True)
        current_batch_size = view1.shape[0]

        optimizer.zero_grad(set_to_none=True)

        projections = model(torch.cat((view1, view2), dim=0))
        z1, z2 = projections.chunk(2, dim=0)

        loss = nt_xent(z1, z2)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * current_batch_size
        sample_count += current_batch_size
        global_step += 1

    return total_loss / sample_count, global_step, first_learning_rate, last_learning_rate


def save_encoder_checkpoint(encoder, path, policy_name, epoch, validation_summary):
    torch.save({
        "encoder_state": encoder.state_dict(),
        "policy_name": policy_name,
        "epoch": epoch,
        "development_summary": validation_summary
        },
        path
      )


def train_policy(policy_name):
    reset_seed()

    loader = make_ssl_loader(policy_name)
    model = SimCLRModel().to(DEVICE)
    optimizer = build_optimizer(model)

    steps_per_epoch = len(loader)
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = WARMUP_EPOCHS * steps_per_epoch
    global_step = 0

    policy_checkpoint_dir = CHECKPOINTS_ROOT / policy_name
    policy_checkpoint_dir.mkdir(parents=True, exist_ok=True,)

    best_checkpoint_path = (policy_checkpoint_dir / "best_encoder.pt")
    final_epoch_checkpoint_path = (policy_checkpoint_dir / "epoch_100_encoder.pt")

    history_rows = []
    probe_rows = []
    best_summary = None
    best_epoch = None

    for epoch in range(1, EPOCHS + 1):
        train_loss, global_step, first_learning_rate, last_learning_rate =  train_epoch(
            model,
            loader,
            optimizer,
            global_step,
            total_steps,
            warmup_steps)

        history_rows.append({
            "epoch": epoch,
            "learning_rate_start": first_learning_rate,
            "learning_rate_end": last_learning_rate,
            "train_nt_xent": train_loss
            })

        print(f"""{policy_name} | epoch {epoch:03d}/{EPOCHS}
            | LR {last_learning_rate:.8f} |  NT-Xent {train_loss:.5f}""")

        if epoch not in PROBE_EPOCHS:
            continue

        summary, development_results, c_search_results = evaluate_development_folds(model.encoder)

        development_results.insert(0, "epoch", epoch)
        development_results.insert(0, "policy_name", policy_name)

        c_search_results.insert(0, "epoch", epoch)
        c_search_results.insert(0, "policy_name", policy_name)

        development_path = (PROBES_ROOT / f"{policy_name}_epoch_{epoch}_development_validation.csv")
        c_search_path = (PROBES_ROOT / f"{policy_name}_epoch_{epoch}_inner_c_search.csv")

        save_csv(development_results, development_path)
        save_csv(c_search_results, c_search_path)

        probe_rows.append({
            "policy_name": policy_name,
            "epoch": epoch,
            **summary,
            "development_validation_path": str(development_path),
            "inner_c_search_path": str(c_search_path)
             })

        if is_better_result(summary,
                            best_summary,
                            loss_key="log_loss_mean",
                            accuracy_key="accuracy_mean",
                            f1_key="f1_macro_mean"):
          best_summary = summary.copy()
          best_epoch = epoch
          save_encoder_checkpoint(model.encoder, best_checkpoint_path, policy_name, epoch, summary)

    history = pd.DataFrame(history_rows)
    history_path = HISTORIES_ROOT / f"{policy_name}.csv"
    save_csv(history, history_path)

    probe_history = pd.DataFrame(probe_rows)
    probe_history_path = (PROBES_ROOT / f"{policy_name}_probe_history.csv")
    save_csv(probe_history, probe_history_path)

    final_epoch_row = probe_history.loc[probe_history["epoch"] == EPOCHS].iloc[0]

    final_epoch_summary = {
    key: float(final_epoch_row[key])
    for key in ("selected_c", "accuracy_mean", "f1_macro_mean", "log_loss_mean")
    }

    save_encoder_checkpoint(model.encoder, final_epoch_checkpoint_path, policy_name, EPOCHS, final_epoch_summary)

    result = {
        "policy_name": policy_name,
        "best_epoch": best_epoch,
        "best_checkpoint_path": str(best_checkpoint_path),
        "epoch_100_checkpoint_path": str(final_epoch_checkpoint_path),
        "history_path": str(history_path),
        "probe_history_path": str(probe_history_path),
    }

    for key, value in best_summary.items():
        result[f"best_{key}"] = value

    for key, value in final_epoch_summary.items():
        result[f"epoch_100_{key}"] = value

    del model
    del optimizer
    del loader

    gc.collect()
    torch.cuda.empty_cache()

    return result

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 10. הרצת ארבע המדיניות ובניית טבלאות ההשוואה


לאחר השלמת כל policy נבנות ונשמרות מחדש שתי טבלאות:

### `research_epoch_100_summary.csv`

השוואת תקציב קבוע בין ארבע המדיניות ב־epoch 100. המיון מתבצע לפי:

1. `epoch_100_accuracy_mean` — מהגבוה לנמוך;
2. `epoch_100_f1_macro_mean` — מהגבוה לנמוך;
3. סדר policy קבוע P1, P2, P3, P4.

### `best_checkpoint_summary.csv`

השוואת ה־checkpoint הטוב ביותר של כל policy. המיון מתבצע לפי:

1. `best_accuracy_mean` — מהגבוה לנמוך;
2. `best_f1_macro_mean` — מהגבוה לנמוך;
3. `best_epoch` — epoch מוקדם יותר;
4. סדר policy קבוע P1, P2, P3, P4.

המיון יציב ודטרמיניסטי. Log Loss נשמר לצורכי ניתוח, אך אינו משתתף
בבחירת המדיניות או ה־checkpoint.

</div>

In [ ]:
def sort_policy_results(frame, loss_column, accuracy_column,f1_column, epoch_column=None):
    ordered = frame.copy()
    ordered["policy_priority"] = ordered["policy_name"].map(POLICY_PRIORITY)

    sort_columns = [loss_column, accuracy_column, f1_column]
    ascending = [True, False, False]

    if epoch_column is not None:
        sort_columns.append(epoch_column)
        ascending.append(True)

    sort_columns.append("policy_priority")
    ascending.append(True)

    return (ordered
            .sort_values(sort_columns, ascending=ascending, kind="stable")
            .drop(columns="policy_priority")
            .reset_index(drop=True)
            )

def select_best_policy(frame):
    candidates = (
        frame
        .assign(policy_priority=frame["policy_name"].map(POLICY_PRIORITY))
        .sort_values("policy_priority", kind="stable")
        .drop(columns="policy_priority")
    )

    best_policy = None

    for _, candidate in candidates.iterrows():
        if is_better_result(candidate, best_policy,loss_key="best_log_loss_mean",accuracy_key="best_accuracy_mean",f1_key="best_f1_macro_mean"):
           best_policy = candidate.copy()

    return best_policy

policy_results = []

for policy_name in POLICY_TRANSFORMS:
    print(f"\nTraining {policy_name} | epochs={EPOCHS} | batch_size={BATCH_SIZE}")

    policy_results.append(train_policy(policy_name))

    completed_results = pd.DataFrame(policy_results)

    final_epoch_summary = sort_policy_results(
    completed_results,
    loss_column="epoch_100_log_loss_mean",
    accuracy_column="epoch_100_accuracy_mean",
    f1_column="epoch_100_f1_macro_mean"
    )

    best_checkpoint_summary = sort_policy_results(
        completed_results,
        loss_column="best_log_loss_mean",
        accuracy_column="best_accuracy_mean",
        f1_column="best_f1_macro_mean",
        epoch_column="best_epoch"
        )

    save_csv(final_epoch_summary, RESULTS_ROOT / "research_epoch_100_summary.csv")
    save_csv(best_checkpoint_summary, RESULTS_ROOT / "best_checkpoint_summary.csv")

print("summary epoch comparison:")
display(final_epoch_summary)

print("Best checkpoint comparison:")
display(best_checkpoint_summary)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 11. נעילת המדיניות, ה־checkpoint ו־\(C\)

השורה הראשונה של `best_checkpoint_summary` מגדירה את הבחירה הסופית.
מתוכה נשלפים:

- `SELECTED_POLICY`;
- `SELECTED_EPOCH`;
- `SELECTED_C`;
- הנתיב ל־checkpoint הנבחר.

קובץ האנקודר הנבחר מועתק ל־:

```text
final_benchmark/selected_encoder.pt
```

בנוסף נשמר `selection.json`, הכולל את המדיניות, ה־epoch, ערך \(C\),
ציוני ה־development, temperature ונתיב האנקודר.

משלב זה כל ההחלטות נעולות. אין בחירה או tuning נוספים על סמך ה־test.

</div>

In [ ]:
selected = select_best_policy(completed_results)

SELECTED_POLICY = selected["policy_name"]
SELECTED_EPOCH = int(selected["best_epoch"])
SELECTED_C = float(selected["best_selected_c"])
SELECTED_CHECKPOINT = Path(selected["best_checkpoint_path"])

FINAL_ENCODER_PATH = FINAL_ROOT / "selected_encoder.pt"
shutil.copy2(SELECTED_CHECKPOINT, FINAL_ENCODER_PATH)

selection = {
    "policy_name": SELECTED_POLICY,
    "epoch": SELECTED_EPOCH,
    "c": SELECTED_C,
    "development_log_loss": float(selected["best_log_loss_mean"]),
    "development_accuracy": float(selected["best_accuracy_mean"]),
    "development_f1_macro": float(selected["best_f1_macro_mean"]),
    "temperature": TEMPERATURE,
    "encoder_path": str(FINAL_ENCODER_PATH)
}

with (FINAL_ROOT / "selection.json").open("w", encoding="utf-8") as file:
    json.dump(selection, file, ensure_ascii=False, indent=2)

display(pd.DataFrame([selection]))

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 12. טעינת האנקודר הנבחר והפקת features

ה־checkpoint הנבחר נטען לתוך `SEResNetEncoder`. האנקודר מועבר למצב
`eval()` וכל הפרמטרים שלו מוקפאים באמצעות `requires_grad_(False)`.

לאחר מכן מופקים פעם אחת features לכל 5,000 תמונות ה־`labeled train`.
הפקה משותפת זו אינה מאחדת את עשרת ה־folds לצורך אימון מסווג. בשלב
ה־benchmark כל fold עדיין נשלף לפי 1,000 האינדקסים הרשמיים שלו.

הפקת features פעם אחת מונעת חישוב חוזר עבור תמונות המופיעות ביותר
מ־fold אחד, תוך שמירה מלאה על ההרכב והחפיפה המקוריים של ה־folds.

</div>

In [ ]:
def load_frozen_encoder(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE,)

    encoder = SEResNetEncoder().to(DEVICE)
    encoder.load_state_dict(checkpoint["encoder_state"])
    encoder.eval()

    for parameter in encoder.parameters():
        parameter.requires_grad_(False)

    return encoder


selected_encoder = load_frozen_encoder(FINAL_ENCODER_PATH)

full_train_features, full_train_labels = extract_features(selected_encoder,labeled_train_dataset)


print("Frozen train features:", full_train_features.shape)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 13. Benchmark סופי של יעילות תוויות

זהו התא הראשון שטוען את ה־`test`. בשלב זה המדיניות, ה־checkpoint ו־\(C\)
כבר נבחרו וננעלו.

לכל אחד מעשרת ניסויי האימון הרשמיים:

1. נשלפים 1,000 ה־features בהתאם לאינדקסים המקוריים של ה־fold.
2. נבנה Pipeline חדש של `StandardScaler → LogisticRegression`.
3. ה־Pipeline משתמש ב־`SELECTED_C` הקבוע ומותאם על כל 1,000 התמונות.
4. המסווג נשמר כ־`fold_01.joblib` עד `fold_10.joblib`.
5. המסווג מוערך על כל 8,000 תמונות ה־test.

אין בשלב זה חלוקת train/validation, Cross-Validation או בחירת
hyperparameter.

### מדדים נשמרים

לכל fold נשמרים:

- Accuracy;
- Macro-F1;
- Macro Precision;
- Log Loss;
- Precision, Recall, F1 ו־support לכל מחלקה;
- מטריצת בלבול בפורמט long;
- נתיב המסווג השמור.

בסיום מחושבים ממוצעים בלבד על עשרת ה־folds:

- mean Accuracy;
- mean Macro-F1;
- mean Macro Precision;
- mean Log Loss.

התוצרים נשמרים בקבצים:

- `label_efficiency_fold_results.csv`;
- `label_efficiency_per_class.csv`;
- `label_efficiency_confusions.csv`;
- `label_efficiency_summary.json`.

אותו test משמש לעשרת הניסויים המוגדרים מראש, אך תוצאותיו אינן משפיעות
על שום החלטה מחקרית.

</div>

In [ ]:
official_test_dataset = datasets.STL10(root=DATA_ROOT, split="test", download=True, transform=EVALUATION_TRANSFORM)

test_features, test_labels = extract_features(selected_encoder, official_test_dataset)

benchmark_rows = []
per_class_frames = []
confusion_frames = []

for fold_index, official_indices in enumerate(STL10_OFFICIAL_FOLDS):
    fold_number = fold_index + 1
    fold_features = full_train_features[official_indices]
    fold_labels = full_train_labels[official_indices]

    classifier = make_linear_classifier(SELECTED_C)
    classifier.fit(fold_features, fold_labels)

    classifier_path = CLASSIFIERS_ROOT / f"fold_{fold_number:02d}.joblib"
    joblib.dump(classifier, classifier_path)

    predictions = classifier.predict(test_features)
    probabilities = classifier.predict_proba(test_features)

    report = classification_report(
        test_labels,
        predictions,
        labels=np.arange(NUMBER_OF_CLASSES),
        target_names=official_test_dataset.classes,
        output_dict=True,
        zero_division=0
    )

    benchmark_rows.append({
        "fold": fold_number,
        "labeled_train_size": len(official_indices),
        "c": SELECTED_C,
        "accuracy": accuracy_score(test_labels, predictions),
        "f1_macro": f1_score(test_labels, predictions, average="macro",zero_division=0),
        "precision_macro": report["macro avg"]["precision"],
        "log_loss": log_loss(test_labels, probabilities, labels=np.arange(NUMBER_OF_CLASSES)),
        "classifier_path": str(classifier_path)
    })

    fold_per_class_rows = []

    for class_name in official_test_dataset.classes:
        class_metrics = report[class_name]
        fold_per_class_rows.append({
            "fold": fold_number,
            "class": class_name,
            "precision": class_metrics["precision"],
            "recall": class_metrics["recall"],
            "f1_score": class_metrics["f1-score"],
            "support": class_metrics["support"]
        })

    per_class_frames.append(pd.DataFrame(fold_per_class_rows))

    confusion = confusion_matrix(test_labels, predictions, labels=np.arange(NUMBER_OF_CLASSES))
    confusion_long = (
        pd.DataFrame(
            confusion,
            index=official_test_dataset.classes,
            columns=official_test_dataset.classes
        )
        .rename_axis("true_class")
        .reset_index()
        .melt(id_vars="true_class", var_name="predicted_class", value_name="count")
    )
    confusion_long.insert(0, "fold", fold_number)
    confusion_frames.append(confusion_long)

benchmark_results = pd.DataFrame(benchmark_rows)
benchmark_per_class = pd.concat(per_class_frames, ignore_index=True)
benchmark_confusions = pd.concat(confusion_frames, ignore_index=True)

benchmark_summary = {
    "policy_name": SELECTED_POLICY,
    "selected_epoch": SELECTED_EPOCH,
    "c": SELECTED_C,
    "number_of_folds": len(STL10_OFFICIAL_FOLDS),
    "labeled_train_size_per_fold": 1000,
    "test_size": len(test_labels),
    "accuracy_mean": float(benchmark_results["accuracy"].mean()),
    "f1_macro_mean": float(benchmark_results["f1_macro"].mean()),
    "precision_macro_mean": float(benchmark_results["precision_macro"].mean()),
    "log_loss_mean": float(benchmark_results["log_loss"].mean())
}

save_csv(benchmark_results, FINAL_ROOT / "label_efficiency_fold_results.csv",)
save_csv(benchmark_per_class, FINAL_ROOT / "label_efficiency_per_class.csv",)
save_csv(benchmark_confusions, FINAL_ROOT / "label_efficiency_confusions.csv",)

with (FINAL_ROOT / "label_efficiency_summary.json").open("w", encoding="utf-8") as file:
    json.dump(benchmark_summary, file, ensure_ascii=False, indent=2)

display(pd.DataFrame([benchmark_summary]))
display(benchmark_results)

<div dir="rtl" style="text-align:right; line-height:1.8; font-family:Arial, sans-serif;">

## 14. אריזת תוצאות ההרצה

כל תיקיית `RESULTS_ROOT` של ההרצה הנוכחית נארזת לקובץ ZIP בשם:

```text
SimCLR_STL10_Label_Efficiency_<RUN_ID>.zip
```

הארכיון כולל את היסטוריות האימון, תוצאות ה־probes, checkpoints,
הבחירה הסופית, עשרת המסווגים הלינאריים ותוצאות ה־benchmark.

קובצי STL-10 עצמם אינם נכללים בארכיון, משום ש־`root_dir` הוא תיקיית
התוצאות של ההרצה ולא תיקיית הפרויקט כולה.

</div>

In [ ]:
archive_path = shutil.make_archive(
    base_name=str(
        Path("/content"/ "SimCLR_STL10_Label_Efficiency_" + RUN_ID),
        format="zip",
        root_dir=RESULTS_ROOT)

print("Results archive:", archive_path)